# CSE 291 / DSC 291 PA3 — Speculative Decoding

In this notebook you will implement and benchmark a single-sequence (batch=1) speculative decoder.

Recap of the algorithm:

1. A small **draft** model proposes `k` tokens autoregressively starting from the current context.
2. The large **target** model verifies the proposal in **one** forward pass (a single batched pass over the `L + k` length sequence).
3. Tokens are accepted greedily up to the first mismatch with the target's argmax. After the first mismatch, the target's own next token is appended and the loop restarts.

Default model pair (public weights, runs on any GPU with >=4 GB VRAM):

- target: `EleutherAI/pythia-1.4b-deduped`
- draft:  `EleutherAI/pythia-160m-deduped`

If you don't have GPU access, the same code paths run on CPU but you won't see a meaningful speedup.

## Setup

In [ ]:
import os
import time
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

## Speculative Decoder

In [ ]:
class SpeculativeDecoder:
    def __init__(self, target_model_name: str, draft_model_name: str, device: str = "cuda"):
        """Initialize the speculative decoder with a target and a draft model."""
        self.device = device
        self.target_model, self.target_tokenizer = self.initialize_target_model(target_model_name)
        self.draft_model, self.draft_tokenizer = self.initialize_draft_model(draft_model_name)

        assert self.target_tokenizer.get_vocab() == self.draft_tokenizer.get_vocab(), (
            "Target and draft must share a vocabulary"
        )

    def initialize_target_model(self, model_name: str):
        """Load the larger target model with caching enabled."""
        print(f"Loading target model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
        ).to(self.device)

        model.eval()
        model.config.use_cache = True

        return model, tokenizer

    def initialize_draft_model(self, model_name: str):
        """Load the smaller draft model."""
        print(f"Loading draft model: {model_name}")
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
        ).to(self.device)

        model.eval()
        model.config.use_cache = True

        return model, tokenizer

    def generate_draft_tokens(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                             num_speculative_tokens: int = 10) -> torch.Tensor:
        """
        Generate `num_speculative_tokens` draft tokens with the draft model.

        Args:
            input_ids: Input token IDs (tensor of shape [1, seq_len]).
            attention_mask: Corresponding attention mask.
            num_speculative_tokens: Number of tokens to speculate.

        Returns:
            Tensor of shape [1, num_speculative_tokens] containing the draft tokens.
        """
        with torch.no_grad():
            output = self.draft_model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=num_speculative_tokens,
                do_sample=False,               # greedy — maximizes acceptance rate
                pad_token_id=self.draft_tokenizer.pad_token_id,
            )
        # Extract only the newly generated tokens (not the input prefix)
        draft_tokens = output[:, input_ids.shape[1]:]
        return draft_tokens

    def verify_tokens_vectorized(self, input_ids: torch.Tensor, draft_tokens: torch.Tensor,
                               attention_mask: torch.Tensor) -> Tuple[List[int], int]:
        """
        Vectorized verification: verify all draft tokens in one forward pass using the target model.

        Args:
            input_ids: The current input token IDs (shape [1, L]).
            draft_tokens: Draft tokens from the draft model (shape [1, k]).
            attention_mask: The current attention mask for input_ids.

        Returns:
            accepted_tokens: List of accepted token IDs.
            accepted_position: Number of draft tokens accepted (not counting bonus token).
        """
        k = draft_tokens.shape[1]
        L = input_ids.shape[1]

        # Build full sequence: [prompt | draft_tokens]
        full_ids  = torch.cat([input_ids, draft_tokens], dim=1)           # (1, L+k)
        full_mask = torch.cat([
            attention_mask,
            torch.ones(1, k, dtype=attention_mask.dtype, device=self.device)
        ], dim=1)

        # Single forward pass through target model
        with torch.no_grad():
            output = self.target_model(full_ids, attention_mask=full_mask)

        logits = output.logits  # (1, L+k, vocab_size)

        # Logits at position j predict token at position j+1.
        # To verify draft token i (at position L+i), use logits[L+i-1].
        # Slice L-1 : L+k  to get predictions for draft[0..k-1] AND the bonus position.
        verify_logits = logits[0, L - 1: L + k, :]    # (k+1, vocab_size)
        predicted     = verify_logits.argmax(dim=-1)  # (k+1,)

        accepted_tokens   = []
        accepted_position = 0

        for i in range(k):
            target_tok = predicted[i].item()
            draft_tok  = draft_tokens[0, i].item()

            if target_tok == draft_tok:
                accepted_tokens.append(draft_tok)
                accepted_position += 1
            else:
                # First mismatch — accept the target's correction and stop
                accepted_tokens.append(target_tok)
                return accepted_tokens, accepted_position

        # All k draft tokens accepted: get one free bonus token from the target.
        # logits[L+k-1] (= predicted[k]) predicts position L+k — one beyond the draft.
        bonus_tok = predicted[k].item()
        accepted_tokens.append(bonus_tok)

        return accepted_tokens, accepted_position

    def speculative_decode(self, prompt: str, max_tokens: int = 100,
                          num_speculative_tokens: int = 8) -> str:
        """
        Main speculative decoding algorithm with vectorized verification.

        Args:
            prompt: Input text.
            max_tokens: Maximum number of tokens to generate (excluding prompt).
            num_speculative_tokens: Number of tokens to speculate per iteration.

        Returns:
            Generated text.
        """
        # Tokenize prompt
        inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)
        prompt_length = input_ids.shape[1]

        # Initialize counters for performance tracking
        total_tokens_generated = prompt_length
        total_draft_tokens_proposed = 0
        total_draft_tokens_accepted = 0
        start_time = time.time()

        while total_tokens_generated - prompt_length < max_tokens:
            remaining = max_tokens - (total_tokens_generated - prompt_length)
            k = min(num_speculative_tokens, remaining)

            # 1. Draft model proposes k tokens
            draft_tokens = self.generate_draft_tokens(input_ids, attention_mask, k)

            # Guard: draft may produce fewer tokens if EOS is hit early
            actual_k = draft_tokens.shape[1]
            if actual_k == 0:
                break
            total_draft_tokens_proposed += actual_k

            # 2. Target model verifies in one pass (returns accepted + optional bonus)
            accepted_tokens, accepted_pos = self.verify_tokens_vectorized(
                input_ids, draft_tokens, attention_mask
            )
            total_draft_tokens_accepted += accepted_pos

            # 3. Append all accepted tokens (including correction or bonus)
            new_ids  = torch.tensor([accepted_tokens], device=self.device)
            input_ids = torch.cat([input_ids, new_ids], dim=1)
            attention_mask = torch.cat([
                attention_mask,
                torch.ones(1, len(accepted_tokens), dtype=attention_mask.dtype, device=self.device)
            ], dim=1)
            total_tokens_generated += len(accepted_tokens)

            # 4. Stop on EOS
            if self.target_tokenizer.eos_token_id in accepted_tokens:
                break

        # Calculate performance metrics
        elapsed_time = time.time() - start_time
        acceptance_rate = (total_draft_tokens_accepted / total_draft_tokens_proposed
                           if total_draft_tokens_proposed > 0 else 0)

        print(f"Generated {total_tokens_generated - prompt_length} tokens in {elapsed_time:.2f} seconds")
        print(f"Tokens per second: {(total_tokens_generated - prompt_length) / elapsed_time:.2f}")
        print(f"Draft token acceptance rate: {acceptance_rate:.2%}")

        self._last_acceptance_rate = acceptance_rate
        return self.target_tokenizer.decode(input_ids[0], skip_special_tokens=True)

    def benchmark(
        self,
        prompt: str,
        max_tokens: int = 100,
        num_runs: int = 3,
        compare_baseline: bool = True,
        num_speculative_tokens: int = 8,
    ) -> Dict:
        results = {
            "speculative": {"times": [], "tokens_per_second": []},
            "baseline": {"times": [], "tokens_per_second": []} if compare_baseline else None,
        }

        for _ in range(num_runs):
            t0 = time.time()
            output = self.speculative_decode(prompt, max_tokens=max_tokens,
                                             num_speculative_tokens=num_speculative_tokens)
            elapsed = time.time() - t0
            prompt_len = len(self.target_tokenizer(prompt)["input_ids"])
            output_tokens = len(self.target_tokenizer.encode(output)) - prompt_len
            results["speculative"]["times"].append(elapsed)
            results["speculative"]["tokens_per_second"].append(output_tokens / elapsed)
            results["speculative"].setdefault("acceptance_rates", []).append(
                getattr(self, "_last_acceptance_rate", 0.0))

        if compare_baseline:
            for _ in range(num_runs):
                inputs = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
                input_ids = inputs["input_ids"].to(self.device)
                attention_mask = inputs["attention_mask"].to(self.device)
                t0 = time.time()
                with torch.no_grad():
                    output_ids = self.target_model.generate(
                        input_ids,
                        attention_mask=attention_mask,
                        max_length=input_ids.shape[1] + max_tokens,
                        do_sample=False,
                        pad_token_id=self.target_tokenizer.pad_token_id,
                    )
                elapsed = time.time() - t0
                output_tokens = output_ids.shape[1] - input_ids.shape[1]
                results["baseline"]["times"].append(elapsed)
                results["baseline"]["tokens_per_second"].append(output_tokens / elapsed)

        for method in results:
            if results[method] is not None:
                results[method]["avg_time"] = sum(results[method]["times"]) / num_runs
                results[method]["avg_tokens_per_second"] = (
                    sum(results[method]["tokens_per_second"]) / num_runs
                )
                if "acceptance_rates" in results[method]:
                    results[method]["avg_acceptance_rate"] = (
                        sum(results[method]["acceptance_rates"]) / num_runs)
        if compare_baseline:
            results["speedup"] = (
                results["baseline"]["avg_time"] / results["speculative"]["avg_time"]
            )
            results["latency_reduction"] = (
                1 - results["speculative"]["avg_time"] / results["baseline"]["avg_time"]
            ) * 100
        return results

## Test

In [ ]:
target_model_name = "EleutherAI/pythia-1.4b-deduped"
draft_model_name = "EleutherAI/pythia-160m-deduped"

decoder = SpeculativeDecoder(
    target_model_name=target_model_name,
    draft_model_name=draft_model_name,
    device="cuda" if torch.cuda.is_available() else "cpu",
)

test_prompts = [
    "The future of artificial intelligence is",
    "Write a short story about a robot learning to feel emotions:",
    "Write the lyrics to the song 'Happy Birthday'."
]

for i, prompt in enumerate(test_prompts):
    print(f"\nBenchmarking Prompt {i+1}: {prompt}")
    results = decoder.benchmark(prompt=prompt, max_tokens=100, num_runs=3, compare_baseline=True)
    print(f"  Speculative: {results['speculative']['avg_time']:.2f}s, "
          f"{results['speculative']['avg_tokens_per_second']:.2f} tok/s")
    if results['baseline'] is not None:
        print(f"  Baseline:    {results['baseline']['avg_time']:.2f}s, "
              f"{results['baseline']['avg_tokens_per_second']:.2f} tok/s")
        print(f"  Speedup: {results['speedup']:.2f}x  |  Latency reduction: {results['latency_reduction']:.2f}%")

Loading target model: EleutherAI/pythia-1.4b-deduped


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/396 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/375M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Benchmarking Prompt 1: The future of artificial intelligence is
Generated 100 tokens in 2.27 seconds
Tokens per second: 44.07
Draft token acceptance rate: 91.67%
Generated 100 tokens in 1.25 seconds
Tokens per second: 80.19
Draft token acceptance rate: 91.67%
Generated 100 tokens in 1.22 seconds
Tokens per second: 81.99
Draft token acceptance rate: 91.67%
  Speculative: 1.58s, 68.64 tok/s
  Baseline:    2.06s, 49.34 tok/s
  Speedup: 1.30x  |  Latency reduction: 23.05%

Benchmarking Prompt 2: Write a short story about a robot learning to feel emotions:
Generated 101 tokens in 1.33 seconds
Tokens per second: 75.71
Draft token acceptance rate: 100.00%
Generated 101 tokens in 1.19 seconds
Tokens per second: 84.96
Draft token acceptance rate: 100.00%
Generated 101 tokens in 1.14 seconds
Tokens per second: 88.31
Draft token acceptance rate: 100.00%
  Speculative: 1.22s, 82.92 tok/s
  Baseline:    1.85s, 53.96 tok/s
  Speedup: 1.51x  |  Latency reduction: 33.99%

Benchmarking Prompt 3: Write

## Part 3.3 — Sweep `num_speculative_tokens`

In [ ]:
# Part 3.3 — Sweep num_speculative_tokens
sweep_prompt = "The future of artificial intelligence is"
sweep_results = {}

for k in [2, 4, 8, 16]:
    print(f"\n--- k = {k} ---")
    res = decoder.benchmark(
        prompt=sweep_prompt,
        max_tokens=100,
        num_runs=3,
        compare_baseline=(k == 2),  # run baseline only once for reference
        num_speculative_tokens=k,
    )
    sweep_results[k] = res
    spec = res["speculative"]
    if k == 2:
        baseline_tps = res["baseline"]["avg_tokens_per_second"]
        baseline_time = res["baseline"]["avg_time"]
        print(f"  Baseline: {baseline_time:.2f}s, {baseline_tps:.2f} tok/s")
    print(f"  k={k}: {spec['avg_time']:.2f}s, {spec['avg_tokens_per_second']:.2f} tok/s")

# Print summary table
print("\n" + "="*70)
print("Sweep Summary (prompt: 'The future of artificial intelligence is')")
print(f"{'k':>4}  {'tok/s':>10}  {'acceptance':>12}  {'speedup vs baseline':>20}")
baseline_time_ref = sweep_results[2]["baseline"]["avg_time"]
for k, res in sweep_results.items():
    tps  = res["speculative"]["avg_tokens_per_second"]
    spd  = baseline_time_ref / res["speculative"]["avg_time"]
    acc  = res["speculative"].get("avg_acceptance_rate", float("nan"))
    print(f"{k:>4}  {tps:>10.2f}  {acc:>11.2%}  {spd:>20.2f}x")


--- k = 2 ---
Generated 100 tokens in 1.50 seconds
Tokens per second: 66.47
Draft token acceptance rate: 97.06%
Generated 100 tokens in 1.44 seconds
Tokens per second: 69.51
Draft token acceptance rate: 97.06%
Generated 100 tokens in 1.68 seconds
Tokens per second: 59.49
Draft token acceptance rate: 97.06%
  Baseline: 2.01s, 50.37 tok/s
  k=2: 1.54s, 65.12 tok/s

--- k = 4 ---
Generated 101 tokens in 1.30 seconds
Tokens per second: 77.75
Draft token acceptance rate: 95.24%
Generated 101 tokens in 1.29 seconds
Tokens per second: 78.47
Draft token acceptance rate: 95.24%
Generated 101 tokens in 1.33 seconds
Tokens per second: 76.10
Draft token acceptance rate: 95.24%
  k=4: 1.31s, 77.38 tok/s

--- k = 8 ---
Generated 100 tokens in 1.21 seconds
Tokens per second: 82.67
Draft token acceptance rate: 91.67%
Generated 100 tokens in 1.52 seconds
Tokens per second: 66.00
Draft token acceptance rate: 91.67%
Generated 100 tokens in 1.62 seconds
Tokens per second: 61.79
Draft token acceptance rat

## Bonus 3.B — Tree speculation or n-gram lookup decoding (10 pts)

Implement one stronger speculative-decoding variant and benchmark it
against the baseline:

- **Tree / multi-branch speculation** (Medusa / EAGLE-2 style): verify
  several candidate continuations in a single target forward pass.
- **N-gram lookup decoding** (Prompt Lookup Decoding): draft the next
  tokens from an n-gram cache built over the running sequence instead of
  (or in addition to) the draft model.

Re-run the benchmark with your bonus decoder and report the speedup and
acceptance rate in your write-up. See the bonus rubric in `../README.md`.

In [ ]:
class NGramLookupDecoder(SpeculativeDecoder):
    """
    Bonus 3.B — N-gram Lookup Decoding combined with the standard draft model.

    At each decoding step, we first search the *running context* for the last
    `ngram_size` tokens. If a match is found, the tokens that followed in the
    context are used as draft tokens — no model inference needed, and the target
    accepts them at near-100% when the text is repetitive. When no match is
    found we fall back to the draft model transparently, so the decoder is
    strictly at least as good as the baseline speculative decoder.

    Why the acceptance rate changes
    --------------------------------
    For repetitive prompts (e.g. 'Happy Birthday to you...') the n-gram draft
    tokens are *exact* copies of text the target model already produced
    deterministically. Because the target is greedy, it must agree with its own
    prior output, so those positions are accepted with probability 1 — pushing
    acceptance rate toward 100%. For novel text the n-gram cache is cold; the
    decoder falls back to the draft model and achieves the same acceptance rate
    as the baseline speculative decoder (~91-93%).
    """

    # ------------------------------------------------------------------
    # Construction helpers
    # ------------------------------------------------------------------
    def __init__(self, target_model_name: str, draft_model_name: str,
                 device: str = "cuda", ngram_size: int = 4):
        super().__init__(target_model_name, draft_model_name, device)
        self.ngram_size = ngram_size

    @classmethod
    def from_decoder(cls, base: "SpeculativeDecoder", ngram_size: int = 4):
        """
        Build an NGramLookupDecoder that *reuses* an already-loaded
        SpeculativeDecoder's models — no double-loading, no extra GPU memory.
        """
        obj = cls.__new__(cls)
        obj.device           = base.device
        obj.target_model     = base.target_model
        obj.target_tokenizer = base.target_tokenizer
        obj.draft_model      = base.draft_model
        obj.draft_tokenizer  = base.draft_tokenizer
        obj.ngram_size       = ngram_size
        return obj

    # ------------------------------------------------------------------
    # N-gram lookup
    # ------------------------------------------------------------------
    def _ngram_lookup(self, context: list, num_tokens: int) -> Optional[List[int]]:
        """
        Scan the context backwards for the most-recent occurrence of the last
        `ngram_size` tokens. Returns the `num_tokens` tokens that followed, or
        None when no full-length match exists.
        """
        n = self.ngram_size
        if len(context) < n + 1:
            return None
        query = tuple(context[-n:])
        for i in range(len(context) - n - 1, -1, -1):
            if tuple(context[i: i + n]) == query:
                following = context[i + n: i + n + num_tokens]
                if len(following) == num_tokens:
                    return following
        return None

    # ------------------------------------------------------------------
    # Core decode loop
    # ------------------------------------------------------------------
    def speculative_decode_ngram(self, prompt: str, max_tokens: int = 100,
                                  num_speculative_tokens: int = 8) -> str:
        """N-gram lookup + draft-model speculative decoding."""
        inputs         = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
        input_ids      = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)
        prompt_length  = input_ids.shape[1]

        total_generated = prompt_length
        total_proposed  = 0
        total_accepted  = 0
        ngram_hits      = 0
        ngram_attempts  = 0
        start_time      = time.time()

        while total_generated - prompt_length < max_tokens:
            remaining = max_tokens - (total_generated - prompt_length)
            k = min(num_speculative_tokens, remaining)

            # ---- Try n-gram lookup first (zero inference cost) ----
            context      = input_ids[0].tolist()
            ngram_tokens = self._ngram_lookup(context, k)
            ngram_attempts += 1
            if ngram_tokens is not None:
                ngram_hits  += 1
                draft_tokens = torch.tensor([ngram_tokens], device=self.device)
            else:
                draft_tokens = self.generate_draft_tokens(input_ids, attention_mask, k)

            actual_k = draft_tokens.shape[1]
            if actual_k == 0:
                break
            total_proposed += actual_k

            # ---- One target forward pass verifies all draft tokens ----
            accepted_tokens, accepted_pos = self.verify_tokens_vectorized(
                input_ids, draft_tokens, attention_mask
            )
            total_accepted += accepted_pos

            # ---- Append and advance ----
            new_ids        = torch.tensor([accepted_tokens], device=self.device)
            input_ids      = torch.cat([input_ids, new_ids], dim=1)
            attention_mask = torch.cat([
                attention_mask,
                torch.ones(1, len(accepted_tokens),
                           dtype=attention_mask.dtype, device=self.device)
            ], dim=1)
            total_generated += len(accepted_tokens)

            if self.target_tokenizer.eos_token_id in accepted_tokens:
                break

        elapsed       = time.time() - start_time
        tokens_out    = total_generated - prompt_length
        accept_rate   = total_accepted / total_proposed if total_proposed  > 0 else 0.0
        ngram_hit_rt  = ngram_hits     / ngram_attempts if ngram_attempts  > 0 else 0.0

        print(f"Generated {tokens_out} tokens in {elapsed:.2f}s")
        print(f"Tokens per second:     {tokens_out / elapsed:.2f}")
        print(f"Draft acceptance rate: {accept_rate:.2%}")
        print(f"N-gram hit rate:       {ngram_hit_rt:.2%}")
        return self.target_tokenizer.decode(input_ids[0], skip_special_tokens=True)

    # ------------------------------------------------------------------
    # Benchmark helper
    # ------------------------------------------------------------------
    def benchmark_ngram(self, prompt: str, max_tokens: int = 100,
                         num_runs: int = 3, k: int = 8) -> Dict:
        """Run n-gram speculative decoder and baseline each num_runs times."""
        # ---- speculative runs ----
        spec_times, spec_tps = [], []
        for _ in range(num_runs):
            t0      = time.time()
            out     = self.speculative_decode_ngram(prompt, max_tokens=max_tokens,
                                                    num_speculative_tokens=k)
            elapsed = time.time() - t0
            p_len   = len(self.target_tokenizer(prompt)["input_ids"])
            toks    = len(self.target_tokenizer.encode(out)) - p_len
            spec_times.append(elapsed)
            spec_tps.append(toks / elapsed)

        # ---- baseline runs (target-only greedy) ----
        base_times, base_tps = [], []
        for _ in range(num_runs):
            inputs  = self.target_tokenizer(prompt, return_tensors="pt", padding=True)
            iids    = inputs["input_ids"].to(self.device)
            mask    = inputs["attention_mask"].to(self.device)
            t0      = time.time()
            with torch.no_grad():
                out_ids = self.target_model.generate(
                    iids, attention_mask=mask,
                    max_length=iids.shape[1] + max_tokens,
                    do_sample=False,
                    pad_token_id=self.target_tokenizer.pad_token_id,
                )
            elapsed = time.time() - t0
            base_times.append(elapsed)
            base_tps.append((out_ids.shape[1] - iids.shape[1]) / elapsed)

        avg_spec = sum(spec_times) / num_runs
        avg_base = sum(base_times) / num_runs
        return {
            "spec_avg_time": avg_spec,
            "spec_avg_tps":  sum(spec_tps) / num_runs,
            "base_avg_time": avg_base,
            "base_avg_tps":  sum(base_tps) / num_runs,
            "speedup":       avg_base / avg_spec,
        }


In [ ]:
# ── Bonus 3.B benchmark ──────────────────────────────────────────────────────
# Reuses the `decoder` object created in the Test cell above.
# No models are reloaded — NGramLookupDecoder.from_decoder() borrows them.

ngram_decoder = NGramLookupDecoder.from_decoder(decoder, ngram_size=4)

bonus_prompts = [
    "Write the lyrics to the song 'Happy Birthday'.",          # repetitive — best n-gram case
    "The future of artificial intelligence is",                # open-ended text
    "Write a short story about a robot learning to feel emotions:",  # narrative
]

print("=" * 70)
print("Bonus 3.B — N-gram Lookup Decoding  (ngram_size=4, k=8, 3 runs each)")
print("=" * 70)

bonus_results = {}
for prompt in bonus_prompts:
    print(f"\nPrompt: {prompt}")
    res = ngram_decoder.benchmark_ngram(prompt, max_tokens=100, num_runs=3, k=8)
    bonus_results[prompt] = res
    print(f"  N-gram spec: {res['spec_avg_time']:.2f}s, {res['spec_avg_tps']:.2f} tok/s")
    print(f"  Baseline:    {res['base_avg_time']:.2f}s, {res['base_avg_tps']:.2f} tok/s")
    print(f"  Speedup:     {res['speedup']:.2f}x")

print("\n" + "=" * 70)
print("Summary")
print(f"  {'Prompt':<52}  {'Speedup':>8}  {'Base tok/s':>10}  {'N-gram tok/s':>12}")
print("  " + "-" * 68)
for p, r in bonus_results.items():
    print(f"  {p[:52]:<52}  {r['speedup']:8.2f}x  "
          f"{r['base_avg_tps']:10.2f}  {r['spec_avg_tps']:12.2f}")


Bonus 3.B — N-gram Lookup Decoding  (ngram_size=4, k=8, 3 runs each)

Prompt: Write the lyrics to the song 'Happy Birthday'.
Generated 101 tokens in 0.43s
Tokens per second:     234.60
Draft acceptance rate: 93.68%
N-gram hit rate:       83.33%
Generated 101 tokens in 0.43s
Tokens per second:     235.99
Draft acceptance rate: 93.68%
N-gram hit rate:       83.33%
Generated 101 tokens in 0.44s
Tokens per second:     228.66
Draft acceptance rate: 93.68%
N-gram hit rate:       83.33%
  N-gram spec: 0.43s, 232.60 tok/s
  Baseline:    1.99s, 50.59 tok/s
  Speedup:     4.58x

Prompt: The future of artificial intelligence is
Generated 100 tokens in 0.61s
Tokens per second:     165.26
Draft acceptance rate: 91.67%
N-gram hit rate:       83.33%
Generated 100 tokens in 0.57s
Tokens per second:     174.40
Draft acceptance rate: 91.67%
N-gram hit rate:       83.33%
Generated 100 tokens in 0.42s
Tokens per second:     238.54
Draft acceptance rate: 91.67%
N-gram hit rate:       83.33%
  N-gram spec: 